# SQuADGoals: train Match-LSTM on Colab

Trains the Match-LSTM + Answer Pointer model on SQuAD v1.1, then scores it on the dev set.

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Run all cells. Data download and preprocessing take about 15–20 minutes. Training is 10 epochs.
3. Checkpoints go to Google Drive, so if Colab disconnects, run all cells again: training resumes from the last checkpoint.

In [ ]:
!nvidia-smi

## Setup

If the repo is private, add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar).

In [ ]:
import os

BRANCH = "claude/beautiful-newton-6tdw2n"
REPO = "github.com/a10y/squadgoals.git"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if not os.path.exists("/content/squadgoals"):
    !git clone -q -b {BRANCH} {url} /content/squadgoals
%cd /content/squadgoals
!git pull -q
!pip install -q uv

In [ ]:
# Keep checkpoints and processed data on Google Drive, so they survive a disconnect.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/squadgoals"
SAVE_DIR = f"{DRIVE_DIR}/save"
os.makedirs(SAVE_DIR, exist_ok=True)

## Data

In [ ]:
# Downloads SQuAD v1.1 and GloVe 840B (about 2 GB), then builds the .npz files.
# Processed files are cached on Drive, so this only runs once.
if os.path.exists(f"{DRIVE_DIR}/data/squad/train.npz"):
    !rm -rf data && cp -r {DRIVE_DIR}/data data
else:
    !./get_started.sh
    !uv run python preprocess.py
    !mkdir -p {DRIVE_DIR}/data/squad
    !cp data/squad/*.npz data/squad/*.npy data/squad/vocab.dat data/squad/dev.qid data/squad/dev-v1.1.json {DRIVE_DIR}/data/squad/
!ls -lh data/squad

## Train

10 epochs, learning rate halves after each epoch. Adds `--resume` when a checkpoint exists.

In [ ]:
import glob
resume = "--resume" if glob.glob(f"{SAVE_DIR}/ckpt-*.index") else ""
!uv sync -q && ./run_train.sh --save_dir={SAVE_DIR} {resume}

## Evaluate on the SQuAD dev set

In [ ]:
!uv run python answer.py --checkpoint={SAVE_DIR}/model_best --out=predictions.json
!uv run python evaluate.py data/squad/dev-v1.1.json predictions.json | tee {DRIVE_DIR}/dev_scores.json

In [ ]:
import json
scores = json.load(open(f"{DRIVE_DIR}/dev_scores.json"))
preds = json.load(open("predictions.json"))
empty = sum(1 for v in preds.values() if not v.strip())

print(f"{'':38s} {'EM':>6s} {'F1':>6s}")
print(f"{'This run (dev)':38s} {scores['exact_match']:6.1f} {scores['f1']:6.1f}")
print(f"{'Old TF1 version (hidden test)':38s} {41.4:6.1f} {53.5:6.1f}")
print(f"{'Paper, Match-LSTM boundary (dev)':38s} {61.1:6.1f} {71.2:6.1f}")
print(f"{'Logistic regression baseline (dev)':38s} {40.0:6.1f} {51.0:6.1f}")
print(f"\nEmpty answers: {empty} of {len(preds)}")